## **Dependencias**

In [22]:
!pip install -q agentpy numpy matplotlib seaborn


[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: C:\Python313\python.exe -m pip install --upgrade pip


## **Imports y estilo**

In [23]:
import agentpy as ap
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
sns.set(context="notebook", style="whitegrid")

## **Parámetros**

In [24]:

params = {
    'steps': 200,
    'green': 20,
    'yellow': 3,
    'all_red': 2,
    'lambda_N': 0.08,
    'lambda_S': 0.12,
    'lambda_E': 0.05,
    'lambda_W': 0.15,
    'L': 100.0,
    'w': 3.5,
    'r_in': 20.0,
    'r_out': 30.0,
    'v_free': 4.0,
    'headway': 16.0,
    'n_entries': 4,
}

## **Controlador de semáforos**

class FourWaySignals(ap.Agent):
    """Control adaptativo: fase NS y fase EW, con subestados G/Y/AR."""

    def setup(self, green_ns, green_ew, yellow, all_red):
        self.g_ns, self.g_ew = int(green_ns), int(green_ew)
        self.y, self.ar = int(yellow), int(all_red)
        self.phase = 0          # 0 = NS verde, 1 = EW verde
        self.sub = 'G'          # 'G','Y','AR'
        self.t_in = 0
        self.timeline = []
        # Parámetros heurísticos
        self.theta = 3
        self.Gmin = 5
        self.Gmax = 20

    def lights(self):
        L = {d:'R' for d in ['N','S','E','W']}
        if self.phase == 0:
            L['N'] = L['S'] = self.sub
        else:
            L['E'] = L['W'] = self.sub
        return L

    @property
    def green_dirs(self):
        if self.sub != 'G': return set()
        return {'N','S'} if self.phase == 0 else {'E','W'}

    def get_queues(self):
        # Calcula la cantidad de autos esperando en cada dirección
        m = self.model
        Q = {'N':0, 'S':0, 'E':0, 'W':0}
        for c in m.cars:
            if c.state == 'stop':
                Q[c.origin] += 1
        return Q

    def step(self):
        self.timeline.append((self.model.t, self.lights()))
        if self.sub == 'G':
            Q = self.get_queues()
            sumQ_NS = Q['N'] + Q['S']
            sumQ_EW = Q['E'] + Q['W']
            # Heurística adaptativa
            if self.phase == 0:
                if sumQ_NS > sumQ_EW + self.theta and self.t_in < self.Gmax:
                    self.t_in += 1  # Extiende verde NS
                elif self.t_in >= self.Gmin:
                    self.sub, self.t_in = 'Y', 0  # Cambia a amarillo
                else:
                    self.t_in += 1
            else:
                if sumQ_EW > sumQ_NS + self.theta and self.t_in < self.Gmax:
                    self.t_in += 1  # Extiende verde EW
                elif self.t_in >= self.Gmin:
                    self.sub, self.t_in = 'Y', 0
                else:
                    self.t_in += 1
        elif self.sub == 'Y' and self.t_in >= self.y:
            self.sub, self.t_in = 'AR', 0
        elif self.sub == 'AR' and self.t_in >= self.ar:
            self.phase = 1 - self.phase
            self.sub, self.t_in = 'G', 0
        else:
            self.t_in += 1

In [25]:
class RoundaboutSignals(ap.Agent):
    """Controlador de semáforos para rotonda: un semáforo por entrada, con heurística adaptativa y tiempos mínimo/máximo de rojo."""

    def setup(self, green, yellow, all_red, n_entries=4):
        self.entries = ['N', 'S', 'E', 'W'][:n_entries]
        self.green = int(green)
        self.yellow = int(yellow)
        self.all_red = int(all_red)
        self.lights_state = {d: {'state': 'G', 't_in': 0, 'red_time': 0, 'last_queue_check': 0} for d in self.entries}
        self.theta = 2
        self.Gmin = 3
        self.Gmax = 14
        self.Rmin = 10
        self.Rmax_base = 30
        self.queue_check_interval = 5
        self.timeline = []

    def lights(self):
        return {d: self.lights_state[d]['state'] for d in self.entries}

    def get_queues(self):
        m = self.model
        Q = {d: 0 for d in self.entries}
        for c in m.cars:
            if c.state == 'stop' and getattr(c, 'at_entry', True):
                Q[c.origin] += 1
        return Q

    def can_enter(self, entry):
        m = self.model
        max_in_roundabout = 12
        n_in = sum([getattr(c, 'state', '') == 'in_roundabout' for c in m.cars])
        if n_in >= max_in_roundabout:
            return False
        entry_angles = {'N': np.pi/2, 'E': 0, 'S': 3*np.pi/2, 'W': np.pi}
        theta_entry = entry_angles[entry]
        r = (m.p.r_in + m.p.r_out) / 2
        min_angular_dist = m.p.headway / r
        for c in m.cars:
            if getattr(c, 'state', None) == 'in_roundabout':
                dtheta = (c.theta - theta_entry) % (2*np.pi)
                if 0 < dtheta < min_angular_dist:
                    return False
        return True

    def step(self):
        self.timeline.append((self.model.t, self.lights()))
        Q = self.get_queues()
        for d in self.entries:
            light = self.lights_state[d]
            fila = Q[d]
            Rmax = max(self.Rmin, self.Rmax_base - fila * 2)
            if light['state'] == 'G':
                if light['last_queue_check'] % self.queue_check_interval == 0:
                    check_queue = True
                else:
                    check_queue = False
                if check_queue and Q[d] > self.theta and self.can_enter(d) and light['t_in'] < self.Gmax:
                    light['t_in'] += 1
                elif light['t_in'] >= self.Gmin:
                    light['state'], light['t_in'], light['red_time'] = 'Y', 0, 0
                    light['last_queue_check'] = 0
                else:
                    light['t_in'] += 1
                light['last_queue_check'] += 1
            elif light['state'] == 'Y' and light['t_in'] >= self.yellow:
                light['state'], light['t_in'], light['red_time'] = 'AR', 0, light['red_time']
                light['last_queue_check'] = 0
            elif light['state'] == 'AR' and light['t_in'] >= self.all_red:
                light['state'], light['t_in'], light['red_time'] = 'R', 0, light['red_time']
                light['last_queue_check'] = 0
            elif light['state'] == 'R':
                light['red_time'] += 1
                # CAMBIO: Si hay autos esperando y hay espacio en la rotonda, cambia a verde aunque no haya pasado el rojo máximo
                if light['red_time'] >= self.Rmin and Q[d] > 0 and self.can_enter(d):
                    light['state'], light['t_in'], light['red_time'] = 'G', 0, 0
                    light['last_queue_check'] = 0
                elif light['red_time'] >= Rmax:
                    light['state'], light['t_in'], light['red_time'] = 'G', 0, 0
                    light['last_queue_check'] = 0
            else:
                light['t_in'] += 1

In [26]:

class FixedTimeRoundaboutSignals(ap.Agent):
    """Controlador de semáforos para rotonda con tiempos fijos (sin heurística)."""
    def setup(self, green, yellow, all_red, n_entries=4):
        self.entries = ['N', 'S', 'E', 'W'][:n_entries]
        self.green = int(green)
        self.yellow = int(yellow)
        self.all_red = int(all_red)
        self.lights_state = {d: {'state': 'G', 't_in': 0} for d in self.entries}
        self.timeline = []
    
    def lights(self):
        return {d: self.lights_state[d]['state'] for d in self.entries}
    
    def step(self):
        self.timeline.append((self.model.t, self.lights()))
        for d in self.entries:
            light = self.lights_state[d]
            if light['state'] == 'G':
                if light['t_in'] >= self.green:
                    light['state'], light['t_in'] = 'Y', 0
                else:
                    light['t_in'] += 1
            elif light['state'] == 'Y':
                if light['t_in'] >= self.yellow:
                    light['state'], light['t_in'] = 'AR', 0
                else:
                    light['t_in'] += 1
            elif light['state'] == 'AR':
                if light['t_in'] >= self.all_red:
                    light['state'], light['t_in'] = 'R', 0
                else:
                    light['t_in'] += 1
            elif light['state'] == 'R':
                if light['t_in'] >= self.green:
                    light['state'], light['t_in'] = 'G', 0
                else:
                    light['t_in'] += 1

## **Agente Vehículo**

In [27]:
class Car(ap.Agent):
    """
    Auto para rotonda: espera en la entrada, entra solo con luz verde, sigue trayectoria circular y sale por su destino.
    """

    def setup(self, origin):
        self.origin = origin  # 'N','S','E','W'
        self.state = 'approach'  # 'approach', 'stop', 'in_roundabout', 'exit', 'done'
        self.v = self.model.p.v_free
        self.r_in = self.model.p.r_in
        self.r_out = self.model.p.r_out
        self.r = (self.r_in + self.r_out) / 2  # Radio medio para trayectoria
        self.L = self.model.p.L
        self.w = self.model.p.w
        self.at_entry = True  # True mientras no entra a la rotonda

        # Ángulos de entrada y salida (en radianes)
        entry_angles = {'N': np.pi/2, 'E': 0, 'S': 3*np.pi/2, 'W': np.pi}
        self.theta_entry = entry_angles[origin]
        # Destino aleatorio distinto de origen
        dests = [d for d in ['N','E','S','W'] if d != origin]
        self.destination = np.random.choice(dests)
        self.theta_exit = entry_angles[self.destination]

        # Posición inicial: más lejos de la rotonda
        entry_point = np.array([self.r * np.cos(self.theta_entry), self.r * np.sin(self.theta_entry)])
        direction = np.array([np.cos(self.theta_entry), np.sin(self.theta_entry)])  
        distancia_inicial = self.L * 1.2  
        self.pos = entry_point + direction * distancia_inicial
        self.theta = self.theta_entry  # Ángulo actual (solo cambia en la rotonda)

    def dist_to_entry(self):
        # Distancia a la entrada de la rotonda
        entry_point = np.array([self.r * np.cos(self.theta_entry), self.r * np.sin(self.theta_entry)])
        return np.linalg.norm(self.pos - entry_point)

    def car_ahead(self):
        # Busca el auto más cercano adelante en la misma entrada (en approach o stop)
        my_dist = self.dist_to_entry()
        ahead = None
        min_dist = None
        for c in self.model.cars:
            if c is self: continue
            if c.origin == self.origin and c.state in ['approach', 'stop'] and getattr(c, 'at_entry', True):
                dist = c.dist_to_entry()
                if dist < my_dist:
                    if ahead is None or dist > min_dist:
                        ahead = c
                        min_dist = dist
        return ahead

    def step(self):
        if self.state == 'done':
            return

        # 1. Aproximación a la rotonda
        if self.state == 'approach':
            # Headway: espera si hay un auto adelante muy cerca
            ahead = self.car_ahead()
            if ahead is not None:
                # Si el auto de adelante está demasiado cerca, no avanza
                if self.dist_to_entry() - ahead.dist_to_entry() < self.model.p.headway:
                    return
            # Si está cerca de la entrada, verifica semáforo
            if self.dist_to_entry() < 8.0:
                lights = self.model.ctrl.lights()
                if lights[self.origin] == 'G':
                    # Al entrar a la rotonda, colocar el auto exactamente en el punto de entrada circular
                    self.pos = np.array([self.r * np.cos(self.theta_entry), self.r * np.sin(self.theta_entry)])
                    self.theta = self.theta_entry
                    self.state = 'in_roundabout'
                    self.at_entry = False
                else:
                    self.state = 'stop'
                    return
            else:
                # Avanza hacia la entrada (siempre hacia el centro)
                direction = (np.array([self.r * np.cos(self.theta_entry), self.r * np.sin(self.theta_entry)]) - self.pos)
                direction = direction / np.linalg.norm(direction)
                self.pos += direction * self.v * 1.0  # dt=1s
                return

        # 2. Esperando en la entrada
        if self.state == 'stop':
            # Headway: espera si hay un auto adelante muy cerca
            ahead = self.car_ahead()
            if ahead is not None:
                if self.dist_to_entry() - ahead.dist_to_entry() < self.model.p.headway:
                    return
            lights = self.model.ctrl.lights()
            if lights[self.origin] == 'G':
                # Al entrar a la rotonda, colocar el auto exactamente en el punto de entrada circular
                self.pos = np.array([self.r * np.cos(self.theta_entry), self.r * np.sin(self.theta_entry)])
                self.theta = self.theta_entry
                self.state = 'in_roundabout'
                self.at_entry = False
            else:
                return

        # 3. Dentro de la rotonda: avanza por el círculo
        if self.state == 'in_roundabout':
            # --- Headway angular: no avanzar si hay un auto adelante muy cerca ---
            # Buscar autos en la rotonda (excepto yo) cuyo ángulo esté adelante y cerca
            ahead_cars = []
            for c in self.model.cars:
                if c is self: continue
                if getattr(c, 'state', None) == 'in_roundabout':
                    # Diferencia angular positiva (en sentido antihorario)
                    dtheta = (c.theta - self.theta) % (2*np.pi)
                    if 0 < dtheta < (self.model.p.headway / self.r):
                        ahead_cars.append((dtheta, c))
            if ahead_cars:
                # Hay un auto adelante muy cerca, no avanzar
                return

            # Avanza un ángulo proporcional a la velocidad
            dtheta = self.v / self.r  # radianes por tick
            self.theta += dtheta 
            self.pos = np.array([self.r * np.cos(self.theta), self.r * np.sin(self.theta)])

            # Verifica si llegó a su salida
            # Considera un margen angular pequeño
            if abs((self.theta - self.theta_exit + np.pi) % (2*np.pi) - np.pi) < 0.1:
                self.state = 'exit'

        # 4. Salida: avanza en línea recta hacia afuera
        if self.state == 'exit':
            direction = np.array([np.cos(self.theta_exit), np.sin(self.theta_exit)])
            self.pos += direction * self.v * 1.0
            # Si está suficientemente lejos del centro, termina
            if np.linalg.norm(self.pos) > self.L:
                self.state = 'done'

## **Modelo con arribos Poisson y listas de agentes**

In [28]:
class RoundaboutModel(ap.Model):
    """
    Modelo de rotonda con arribos Poisson, semáforos adaptativos y agentes vehículo.
    """
    def setup(self):
        global max_en_rotonda, suma_en_rotonda, max_fila, suma_fila
        p = self.p
        tipo = getattr(p, 'controller_type', 'heuristic')
        if tipo == 'fixed':
            self.ctrl = FixedTimeRoundaboutSignals(self, p.green, p.yellow, p.all_red, p.n_entries)
        else:
            self.ctrl = RoundaboutSignals(self, p.green, p.yellow, p.all_red, p.n_entries)
        self.cars = ap.AgentList(self, 0, Car)
        self.spawn_counts = {d: 0 for d in ['N', 'S', 'E', 'W']}
        self.log = []

    def headway_ahead(self, me):
        """
        Busca el auto más cercano adelante en la misma trayectoria (solo para la entrada, no en la rotonda).
        """
        if me.state != 'approach':
            return None
        same = [c for c in self.cars if c is not me and c.state == 'approach' and c.origin == me.origin]
        if not same:
            return None
        # El que está más cerca de la entrada pero adelante de mí
        my_dist = me.dist_to_entry()
        ahead = [(c.dist_to_entry(), c) for c in same if c.dist_to_entry() < my_dist]
        if not ahead:
            return None
        return min(ahead, key=lambda x: x[0])[1]

    def spawn_poisson(self, origin, lam):
        k = np.random.poisson(lam)
        for _ in range(k):
            entry_angles = {'N': np.pi/2, 'E': 0, 'S': 3*np.pi/2, 'W': np.pi}
            theta_entry = entry_angles[origin]
            r = (self.p.r_in + self.p.r_out) / 2
            L = self.p.L
            entry_point = np.array([r * np.cos(theta_entry), r * np.sin(theta_entry)])
            direction = np.array([np.cos(theta_entry), np.sin(theta_entry)])  
            distancia_inicial = L * 1.2
            pos_entry = entry_point + direction * distancia_inicial
            too_close = any(
                np.linalg.norm(c.pos - pos_entry) < self.p.headway
                for c in self.cars if hasattr(c, 'pos')
            )
            if not too_close:
                self.cars.append(Car(self, origin=origin))
                self.spawn_counts[origin] += 1

    def step(self):
        global max_en_rotonda, suma_en_rotonda, max_fila, suma_fila, autos_final
        self.spawn_poisson('N', self.p.lambda_N)
        self.spawn_poisson('S', self.p.lambda_S)
        self.spawn_poisson('E', self.p.lambda_E)
        self.spawn_poisson('W', self.p.lambda_W)
        self.ctrl.step()
        self.cars.step()
        self.cars = ap.AgentList(self, [c for c in self.cars if c.state != 'done'], Car)
        # --- Acumuladores de métricas ---
        n_in = sum(getattr(c, 'state', '') == 'in_roundabout' for c in self.cars)
        max_en_rotonda = max(max_en_rotonda, n_in)
        suma_en_rotonda += n_in
        Q = [0,0,0,0]
        for c in self.cars:
            if c.state == 'stop' and getattr(c, 'at_entry', True):
                idx = ['N','S','E','W'].index(c.origin)
                Q[idx] += 1
        fila_max = max(Q)
        max_fila = max(max_fila, fila_max)
        suma_fila += sum(Q)/4
        autos_final = len(self.cars)
        

In [31]:
# --- Variables globales para métricas ---
max_en_rotonda = 0
suma_en_rotonda = 0
max_fila = 0
suma_fila = 0
autos_final = 0

# --- Ejecutar modelo con controlador fijo ---
max_en_rotonda = 0; suma_en_rotonda = 0; max_fila = 0; suma_fila = 0; autos_final = 0
params['controller_type'] = 'fixed'
model = RoundaboutModel(params)
model.setup()
for t in range(params['steps']):
    model.t = t
    model.step()
print('Controlador FIJO:')
print(f'  Promedio en rotonda: {suma_en_rotonda/params["steps"]:.2f}')
print(f'  Máximo en rotonda: {max_en_rotonda}')
print(f'  Promedio fila: {suma_fila/params["steps"]:.2f}')
print(f'  Máxima fila: {max_fila}')
print(f'  Autos en sistema (final): {autos_final}')

# --- Ejecutar modelo con controlador heurístico ---
max_en_rotonda = 0; suma_en_rotonda = 0; max_fila = 0; suma_fila = 0; autos_final = 0
params['controller_type'] = 'heuristic'
model = RoundaboutModel(params)
model.setup()
for t in range(params['steps']):
    model.t = t
    model.step()
print('\nControlador HEURÍSTICO:')
print(f'  Promedio en rotonda: {suma_en_rotonda/params["steps"]:.2f}')
print(f'  Máximo en rotonda: {max_en_rotonda}')
print(f'  Promedio fila: {suma_fila/params["steps"]:.2f}')
print(f'  Máxima fila: {max_fila}')
print(f'  Autos en sistema (final): {autos_final}')

Controlador FIJO:
  Promedio en rotonda: 12.41
  Máximo en rotonda: 38
  Promedio fila: 0.32
  Máxima fila: 1
  Autos en sistema (final): 51

Controlador HEURÍSTICO:
  Promedio en rotonda: 4.35
  Máximo en rotonda: 9
  Promedio fila: 0.38
  Máxima fila: 1
  Autos en sistema (final): 26


## **Función de animación (animation_plot)**

In [ ]:
def draw_roundabout(ax, L, w, r_in, r_out, model):
    ax.clear()
    ax.set_xlim(-L, L)
    ax.set_ylim(-L, L)
    ax.set_aspect('equal')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(f"Rotonda | t = {model.t}s")
    
    # --- Calles de acceso: dos carriles por acceso (entrada y salida), bien separados y del mismo color ---
    entry_angles = {'N': np.pi/2, 'E': 0, 'S': 3*np.pi/2, 'W': np.pi}
    lane_gap = 5 * w  # Aumenta aún más la separación entre los centros de los dos carriles
    lane_length = L - r_out  # largo visible de los carriles desde el borde de la rotonda
    
    for label, theta in entry_angles.items():
        dir_vec = np.array([np.cos(theta), np.sin(theta)])
        perp_vec = np.array([-np.sin(theta), np.cos(theta)])
        entry_center = dir_vec * (r_out + lane_length/2) + perp_vec * (lane_gap/2)
        exit_center = dir_vec * (r_out + lane_length/2) - perp_vec * (lane_gap/2)
        ax.add_patch(plt.Rectangle(
            entry_center - dir_vec * (lane_length/2) - perp_vec * (w/2),
            lane_length, w,
            angle=np.degrees(theta), color='#e0e0e0', zorder=0
        ))
        ax.add_patch(plt.Rectangle(
            exit_center - dir_vec * (lane_length/2) - perp_vec * (w/2),
            lane_length, w,
            angle=np.degrees(theta), color='#e0e0e0', zorder=0
        ))
    
    # --- Rotonda: anillo ---
    circ1 = plt.Circle((0, 0), r_out, color='black', fill=True, zorder=1)
    circ2 = plt.Circle((0, 0), r_in, color='white', fill=True, zorder=2)
    ax.add_patch(circ1)
    ax.add_patch(circ2)
    
    # --- Líneas de alto (en cada entrada, alineadas con el carril de entrada) ---
    for label, theta in entry_angles.items():
        dir_vec = np.array([np.cos(theta), np.sin(theta)])
        perp_vec = np.array([-np.sin(theta), np.cos(theta)])
        stop_center = dir_vec * (r_out + 2) + perp_vec * (lane_gap/2)
        dx, dy = perp_vec
        ax.plot([stop_center[0] - 6*dx, stop_center[0] + 6*dx],
                [stop_center[1] - 6*dy, stop_center[1] + 6*dy],
                color='yellow', lw=2, zorder=3)
    
    # --- Semáforos: círculos en la entrada de cada calle ---
    if hasattr(model, 'ctrl') and hasattr(model.ctrl, 'lights'):
        lights = model.ctrl.lights()
        color_map = {'R': '#d32f2f', 'Y': '#f9a825', 'G': '#388e3c', 'AR': '#000000'}
        for d, theta in entry_angles.items():
            x, y = (r_out + 5) * np.cos(theta), (r_out + 5) * np.sin(theta)
            state = lights.get(d, 'R')
            ax.add_patch(plt.Circle((x, y), 2.0, color=color_map[state], zorder=4))
    
    # --- Etiquetas de entradas ---
    for label, theta in entry_angles.items():
        x, y = (L + 10) * np.cos(theta), (L + 10) * np.sin(theta)
        ax.text(x, y, label, fontsize=16, fontweight='bold', color='black', ha='center', va='center', zorder=10)
    
    # --- Autos: ajustar posición según carril de entrada/salida ---
    if hasattr(model, 'cars') and len(model.cars) > 0:
        xs, ys, cs = [], [], []
        for c in model.cars:
            pos = np.array(c.pos)
            color = '#9c27b0'  # morado por defecto
            if c.state == 'stop':
                color = '#ff9800'
            elif c.state == 'in_roundabout':
                color = '#1976d2'
            elif c.state == 'exit':
                color = '#43a047'
            # Ajuste de posición para carril de entrada/salida
            if c.state in ['approach', 'stop'] and hasattr(c, 'origin'):
                theta = entry_angles.get(c.origin, 0)
                perp_vec = np.array([-np.sin(theta), np.cos(theta)])
                # Carril de entrada: +lane_gap/2 (derecho para sur, izquierdo para norte, etc.)
                pos = pos + perp_vec * (lane_gap/2)
            elif c.state == 'exit' and hasattr(c, 'theta_exit'):
                theta = c.theta_exit
                perp_vec = np.array([-np.sin(theta), np.cos(theta)])
                # Carril de salida: -lane_gap/2
                pos = pos - perp_vec * (lane_gap/2)
            # Si está en la rotonda, no se ajusta
            xs.append(pos[0])
            ys.append(pos[1])
            cs.append(color)
        ax.scatter(xs, ys, s=80, c=cs, edgecolor='k', linewidth=1.0, zorder=5)
    
def my_plot_roundabout(model, ax):
    draw_roundabout(ax, model.p.L, model.p.w, model.p.r_in, model.p.r_out, model)

## **Correr animación**

In [ ]:
fig, ax = plt.subplots(figsize=(6,6))
model = RoundaboutModel(params)
anim = ap.animate(model, fig, ax, my_plot_roundabout)
from IPython.display import HTML
HTML(anim.to_jshtml())
